# Chess Data Cleaning & Feature Engineering

This notebook transforms the raw Chess.com API data into a clean analytics-ready dataset.

Main goals:
- clean timestamps
- standardize game results
- create player-centric metrics
- engineer analytical features
- prepare the dataset for visualization and modeling

## Imports

In [12]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import pycountry

## Load raw dataset

In [13]:
PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import USERNAME

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "raw"
    / USERNAME
    / "games.csv"
)

games = pd.read_csv(RAW_DATA_PATH)

games.head()

,date,url,time_class,time_control,white,black,white_rating,black_rating,white_result,black_result,eco,pgn,opponent,opponent_clean
0,2020-11-10 15:30:24,https://www.chess.com/game/live/5719328865,rapid,600,elmurie,TheMrNoName,213,412,resigned,win,https://www.chess.com/openings/Polish-Opening-...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",TheMrNoName,themrnoname
1,2020-11-10 15:46:19,https://www.chess.com/game/live/5719418873,rapid,600,smackersmashbot,elmurie,216,345,resigned,win,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",smackersmashbot,smackersmashbot
2,2020-11-10 16:09:31,https://www.chess.com/game/live/5719482469,rapid,600,amkh98,elmurie,371,256,win,checkmated,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",amkh98,amkh98
3,2020-11-10 17:39:54,https://www.chess.com/game/live/5720005454,rapid,600,elmurie,callumfindlay4,193,330,checkmated,win,https://www.chess.com/openings/Vienna-Game-Max...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",callumfindlay4,callumfindlay4
4,2020-11-10 17:51:42,https://www.chess.com/game/live/5720050101,rapid,600,callumfindlay4,elmurie,291,277,resigned,win,https://www.chess.com/openings/Caro-Kann-Defen...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",callumfindlay4,callumfindlay4


## Dataset overview

As explained in the previous notebook, the dataset is pretty reliable.

In [14]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21007 entries, 0 to 21006
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   date            21007 non-null  object
 1   url             21007 non-null  object
 2   time_class      21007 non-null  object
 3   time_control    21007 non-null  object
 4   white           21007 non-null  object
 5   black           21007 non-null  object
 6   white_rating    21007 non-null  int64 
 7   black_rating    21007 non-null  int64 
 8   white_result    21007 non-null  object
 9   black_result    21007 non-null  object
 10  eco             21007 non-null  object
 11  pgn             21007 non-null  object
 12  opponent        21007 non-null  object
 13  opponent_clean  21007 non-null  object
dtypes: int64(2), object(12)
memory usage: 2.2+ MB


In [15]:
games.isna().mean().sort_values(
    ascending=False
)

date              0.0
url               0.0
time_class        0.0
time_control      0.0
white             0.0
black             0.0
white_rating      0.0
black_rating      0.0
white_result      0.0
black_result      0.0
eco               0.0
pgn               0.0
opponent          0.0
opponent_clean    0.0
dtype: float64

## Datetime cleaning

Let's split datetime into useful features for time-based analysis 

In [16]:
games["date"] = pd.to_datetime(
    games["date"]
)

games["year"] = games["date"].dt.year
games["month"] = games["date"].dt.month
games["day"] = games["date"].dt.day
games["weekday"] = games["date"].dt.day_name()
games["hour"] = games["date"].dt.hour

## Player perspective engineering

In every observation both players are listed playing as either white or black.

But we want to focus on the target player and be sure on who's who.

### Side played

In [17]:
games["side"] = np.where(
    games["white"].str.lower() == USERNAME,
    "white",
    "black"
)

### Player rating

In [18]:
games["player_rating"] = np.where(
    games["side"] == "white",
    games["white_rating"],
    games["black_rating"]
)

games["opponent_rating"] = np.where(
    games["side"] == "white",
    games["black_rating"],
    games["white_rating"]
)

### Rating difference

Let's keep in mind that in live chess a user gets paired with a similarly rated player 

In [19]:
games["rating_diff"] = (
    games["player_rating"]
    - games["opponent_rating"]
)

## Standardise results

Out of all the different outcomes in the dataset, we want to establish if our target player obtained a win, a draw or a loss

In [20]:
games["player_result"] = np.where(
    games["side"] == "white",
    games["white_result"],
    games["black_result"]
)

games["opponent_result"] = np.where(
    games["side"] == "white",
    games["black_result"],
    games["white_result"]
)

games["player_result"].unique()

array(['resigned', 'win', 'checkmated', 'stalemate', 'timeout', 'agreed',
       'abandoned', 'insufficient', 'timevsinsufficient', 'repetition'],
      dtype=object)

### Mapping

In [21]:
WIN_RESULTS = ["win"]

DRAW_RESULTS = [
    "agreed",
    "stalemate",
    "repetition",
    "insufficient",
    "50move",
    "timevsinsufficient"
]

LOSS_RESULTS = [
    "checkmated",
    "timeout",
    "resigned",
    "lose",
    "abandoned"
]

### Standardised result obtained

In [22]:
def standardize_result(result):

    if result in WIN_RESULTS:
        return "win"

    elif result in DRAW_RESULTS:
        return "draw"

    elif result in LOSS_RESULTS:
        return "loss"

    return "other"


games["result"] = games[
    "player_result"
].apply(standardize_result)

games["result"].unique()

array(['loss', 'win', 'draw'], dtype=object)

## Numerical target

Let's assign a numerical value for every outcome

In [23]:
result_map = {
    "win": 1,
    "draw": 0.5,
    "loss": 0
}

games["score"] = games["result"].map(
    result_map
)

print(games[games["score"].isna()])

Empty DataFrame
Columns: [date, url, time_class, time_control, white, black, white_rating, black_rating, white_result, black_result, eco, pgn, opponent, opponent_clean, year, month, day, weekday, hour, side, player_rating, opponent_rating, rating_diff, player_result, opponent_result, result, score]
Index: []

[0 rows x 27 columns]


## Opponent username

In [24]:
games["opponent"] = np.where(
    games["side"] == "white",
    games["black"],
    games["white"]
)

## Opening extraction

Chess.com API only returns the ECO URL, so let's extract the opening name

In [25]:
games["opening"] = (
    games["eco"]
    .str.split("/")
    .str[-1]
    .str.replace("-", " ")
    .str.title()
)

games["opening"] = (
    games["opening"]
    .str.replace(r'\s\d+\..*', '', regex=True)
    .str.replace(r'\.\.\..*', '', regex=True)
    .str.strip()
)

games["opening"].head(10)

0                               Polish Opening
1    Kings Pawn Opening Kings Knight Variation
2       Kings Pawn Opening Leonardis Variation
3                Vienna Game Max Lange Defense
4                            Caro Kann Defense
5                                  Vienna Game
6       Queens Pawn Opening Chigorin Variation
7                           Kings Pawn Opening
8                 Pirc Defense Maroczy Defense
9                             Sicilian Defense
Name: opening, dtype: object

## Select final columns

In [26]:
final_columns = [
    "date",

    "year",
    "month",
    "day",
    "weekday",
    "hour",

    "side",

    "time_class",
    "time_control",

    "player_rating",
    "opponent_rating",
    "rating_diff",

    "result",
    "player_result",
    "opponent_result",
    "score",

    "opening",
    "opponent",

    "url"
]

games_clean = games[final_columns].copy()

## Final inspection

In [27]:
games_clean.head()

,date,year,month,day,weekday,hour,side,time_class,time_control,player_rating,opponent_rating,rating_diff,result,player_result,opponent_result,score,opening,opponent,url
0,2020-11-10 15:30:24,2020,11,10,Tuesday,15,white,rapid,600,213,412,-199,loss,resigned,win,0.0,Polish Opening,TheMrNoName,https://www.chess.com/game/live/5719328865
1,2020-11-10 15:46:19,2020,11,10,Tuesday,15,black,rapid,600,345,216,129,win,win,resigned,1.0,Kings Pawn Opening Kings Knight Variation,smackersmashbot,https://www.chess.com/game/live/5719418873
2,2020-11-10 16:09:31,2020,11,10,Tuesday,16,black,rapid,600,256,371,-115,loss,checkmated,win,0.0,Kings Pawn Opening Leonardis Variation,amkh98,https://www.chess.com/game/live/5719482469
3,2020-11-10 17:39:54,2020,11,10,Tuesday,17,white,rapid,600,193,330,-137,loss,checkmated,win,0.0,Vienna Game Max Lange Defense,callumfindlay4,https://www.chess.com/game/live/5720005454
4,2020-11-10 17:51:42,2020,11,10,Tuesday,17,black,rapid,600,277,291,-14,win,win,resigned,1.0,Caro Kann Defense,callumfindlay4,https://www.chess.com/game/live/5720050101


In [28]:
games_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21007 entries, 0 to 21006
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             21007 non-null  datetime64[ns]
 1   year             21007 non-null  int32         
 2   month            21007 non-null  int32         
 3   day              21007 non-null  int32         
 4   weekday          21007 non-null  object        
 5   hour             21007 non-null  int32         
 6   side             21007 non-null  object        
 7   time_class       21007 non-null  object        
 8   time_control     21007 non-null  object        
 9   player_rating    21007 non-null  int64         
 10  opponent_rating  21007 non-null  int64         
 11  rating_diff      21007 non-null  int64         
 12  result           21007 non-null  object        
 13  player_result    21007 non-null  object        
 14  opponent_result  21007 non-null  objec

In [29]:
games_clean.describe(include="all")

,date,year,month,day,weekday,hour,side,time_class,time_control,player_rating,opponent_rating,rating_diff,result,player_result,opponent_result,score,opening,opponent,url
count,21007,21007.000000,21007.000000,21007.000000,21007,21007.000000,21007,21007,21007,21007.000000,21007.000000,21007.000000,21007,21007,21007,21007.000000,21007,21007,21007
unique,NaN,NaN,NaN,NaN,7,NaN,2,4,11,NaN,NaN,NaN,3,10,10,NaN,435,20242,21007
top,NaN,NaN,NaN,NaN,Monday,NaN,black,blitz,180,NaN,NaN,NaN,win,win,win,NaN,Four Knights Game Scotch Variation Accepted,ElettricRunner,https://www.chess.com/game/live/5719328865
freq,NaN,NaN,NaN,NaN,3264,NaN,10519,19825,15802,NaN,NaN,NaN,10182,10182,10157,NaN,1127,33,1
mean,2024-06-03 01:08:37.918931712,2023.946875,6.200171,15.700576,NaN,15.218737,NaN,NaN,NaN,627.966725,628.125577,-0.158852,NaN,NaN,NaN,0.500595,NaN,NaN,NaN
min,2020-11-10 15:30:24,2020.000000,1.000000,1.000000,NaN,0.000000,NaN,NaN,NaN,100.000000,100.000000,-821.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
25%,2023-09-06 01:10:40.500000,2023.000000,3.000000,8.000000,NaN,12.000000,NaN,NaN,NaN,559.000000,563.000000,-21.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
50%,2024-10-26 22:38:51,2024.000000,6.000000,16.000000,NaN,16.000000,NaN,NaN,NaN,649.000000,649.000000,-1.000000,NaN,NaN,NaN,0.500000,NaN,NaN,NaN
75%,2025-06-18 13:08:50,2025.000000,9.000000,23.000000,NaN,20.000000,NaN,NaN,NaN,710.000000,711.000000,20.000000,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
max,2026-05-08 23:49:17,2026.000000,12.000000,31.000000,NaN,23.000000,NaN,NaN,NaN,1041.000000,1607.000000,597.000000,NaN,NaN,NaN,1.000000,NaN,NaN,NaN


## Add Opponent's country

In [30]:
RAW_DATA_DIR = PROJECT_ROOT / "dataset" / "raw" / USERNAME
profiles_path = (
    RAW_DATA_DIR
    / "opponent_profiles.csv"
)

opponent_profiles = pd.read_csv(profiles_path)

In [31]:
games_clean["opponent_clean"] = (
    games_clean["opponent"]
    .str.lower()
    .str.strip()
)

games_geo = games_clean.merge(

    opponent_profiles[
        ["opponent_clean", "country_code"]
    ],

    on="opponent_clean",

    how="left"
)

games_geo["country_code"].value_counts()

country_code
US    3046
IN    1964
FR    1042
GB     831
BR     806
      ... 
FK       1
SM       1
DJ       1
GF       1
CV       1
Name: count, Length: 223, dtype: int64

In [32]:
games_geo["country_code"].isna().mean()

np.float64(0.030751654210501262)

# ISO CONVERSION

In [33]:
def get_country_info(code):

    try:

        country = pycountry.countries.get(
            alpha_2=code
        )

        return pd.Series({

            "iso3": country.alpha_3,

            "country_name": country.name
        })

    except:

        return pd.Series({

            "iso3": None,

            "country_name": None
        })

In [34]:
games_geo[
    ["iso3", "country_name"]
] = games_geo[
    "country_code"
].apply(get_country_info)

## Save processed dataset

In [35]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
    / USERNAME
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

games_geo.to_csv(
    PROCESSED_DIR / f"{USERNAME}_games_clean.csv",
    index=False
)

games_geo.to_excel(
    PROCESSED_DIR / f"{USERNAME}_games_clean.xlsx",
    index=False
)

print("Processed dataset saved.")

Processed dataset saved.
